# 01 - Data Processing

This notebook builds analysis-ready lipidomics datasets directly from raw ROSMAP inputs.

## Scientific goal

Prepare a clean sample-by-lipid table with social isolation and neuropathology covariates so downstream models can be run reproducibly.

## Script equivalent

The same workflow is available via `scripts/01_data_processing.py`.


In [ ]:
import pandas as pd

from config import (
    DATA_PROCESSED_DIR,
    FINAL_FORMATTED_FILENAME,
    NORMALIZED_FORMATTED_FILENAME,
    ensure_project_dirs,
)
from src.data_utils import (
    add_individual_id_column,
    add_individual_ids_to_longitudinal,
    attach_covariates,
    drop_optional_columns,
    load_raw_input_tables,
    merge_lipid_id_map,
    normalize_by_internal_standards,
    reshape_lipidomics_matrix,
)

ensure_project_dirs()


## Step 1: Load raw input tables


In [ ]:
tables = load_raw_input_tables()
for name, df in tables.items():
    print(f"{name:28s} -> shape={df.shape}")


## Step 2: Attach `individualID` to longitudinal metadata

The longitudinal table is keyed by `projid`, while sample-level analysis needs `individualID`. We map through the clinical table.


In [ ]:
longitudinal_with_ids = add_individual_ids_to_longitudinal(
    tables["metadata_longitudinal"],
    tables["metadata_clinical"],
)
print("Longitudinal table with IDs:", longitudinal_with_ids.shape)
longitudinal_with_ids[["projid", "individualID"]].head()


## Step 3: Reshape lipidomics matrix to sample-by-lipid format


In [ ]:
sample_df = reshape_lipidomics_matrix(tables["lipidomics_raw"])
print("Reshaped sample table:", sample_df.shape)
sample_df.iloc[:3, :8]


## Step 4: Add IDs, normalize by internal standards, and attach covariates

This mirrors the intended preprocessing logic from the original workflow.


In [ ]:
sample_df = add_individual_id_column(sample_df)
sample_df = normalize_by_internal_standards(sample_df)
sample_df = merge_lipid_id_map(sample_df, tables["lipid_individual_map"])
sample_df = attach_covariates(sample_df, longitudinal_with_ids)
sample_df = drop_optional_columns(sample_df, columns=["Net_SI", "Sex"])
print("Post-merge sample table:", sample_df.shape)
sample_df[["label", "individualID", "SI_avg", "niareagansc", "msex", "age_death"]].head()


## Step 5: Write normalized and final analysis tables


In [ ]:
normalized_path = DATA_PROCESSED_DIR / NORMALIZED_FORMATTED_FILENAME
final_path = DATA_PROCESSED_DIR / FINAL_FORMATTED_FILENAME

sample_df.to_csv(normalized_path, index=False)

final_df = sample_df.copy()
final_df = final_df.dropna(subset=["individualID"])
final_df = final_df.dropna(subset=["SI_avg"])
final_df = final_df.drop_duplicates(subset=["label"], keep="first")
final_df.to_csv(final_path, index=False)

print(f"Normalized dataset: {normalized_path} | shape={sample_df.shape}")
print(f"Final dataset:      {final_path} | shape={final_df.shape}")
final_df.head(3)


## Next notebook

Run `notebooks/02_quality_control.ipynb` to check sample outliers and distributional assumptions.
